<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/feat%2Funet-training-diagnosis/notebooks/3_2_unet_training_diagnosis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4.2 - Depurar el entrenamiento del U-Net

**Objetivo.** Determinar por que el mejor checkpoint del U-Net es el de la epoca 1.

**Hipotesis a contrastar.** La etiqueta es casi toda estructura y el modelo aprende la constante.

**Punto de partida.** El entrenamiento de `3_1_unet_baseline_training.ipynb` dejo este registro:

```
epoca   1 | loss 1.1208 | val mIoU 0.7025 | val Dice 0.7485   <- mejor checkpoint
epoca   5 | loss 0.8916 | val mIoU 0.6463 | val Dice 0.7061
epoca  20 | loss 0.8575 | val mIoU 0.6872 | val Dice 0.7366
epoca  40 | loss 0.8132 | val mIoU 0.6777 | val Dice 0.7270
train: 84 frames | val: 21 frames | U-Net 31.04M parametros
```

La perdida baja de forma monotona de 1,12 a 0,81 y el Dice de validacion nunca vuelve al valor de
la epoca 1.

**Estructura del notebook.** La parte (b) va primero porque no necesita GPU y, por si sola, decide
la hipotesis. Las partes (a) y (c) requieren Colab con GPU.

| seccion | parte de la tarea | necesita GPU |
|---|---|---|
| 2 | (b) linea base trivial sobre el mismo split | no |
| 3 | hallazgo colateral: auditoria del split | no |
| 4 | (a) curvas de perdida y metricas de train y val | si |
| 5 | (c) variacion de perdida y de schedule | si |
| 6 | conclusion | - |

## 1. Entorno

In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

REPO_URL = "https://github.com/anthony-yt/deepwater-image-segmentation-classical-quantum.git"
BRANCH = "feat/unet-training-diagnosis"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Under_Water_Dataset")
    if not Path("src/training").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                        REPO_URL, "/content/repo"], check=True)
        os.chdir("/content/repo")
else:
    DRIVE_ROOT = Path(os.environ.get("DW_DATA_ROOT", "Under_Water_Dataset")).resolve()
sys.path.insert(0, str(Path.cwd()))

PROCESSED = DRIVE_ROOT / "Dataset_Processed"
MMSEG_ROOT = PROCESSED / "mmseg_pipes"
OUT_DIR = PROCESSED / "unet_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMG_SIZE = 512
BATCH = 4
EPOCHS = 40
LR = 1e-4
FAST = False   # True: 3 epocas, para comprobar que el notebook corre de punta a punta

print("datos :", MMSEG_ROOT)
print("salida:", OUT_DIR)

## 2. Parte (b) - la linea base trivial sobre el mismo split

La pregunta es si el Dice de 0,7485 es bueno. Sin una referencia no se puede saber, asi que se
calcula lo que obtendria un modelo que no aprende nada y predice que **todos** los pixeles son
estructura.

Esta seccion lee solo los PNG de anotacion: no hace falta GPU, ni torch, ni reentrenar.

In [ ]:
def read_binary_mask(path) -> np.ndarray:
    """Mascara en {0,1}. convert("L") aplica la paleta; cv2.IMREAD_GRAYSCALE no lo hace."""
    grey = np.asarray(Image.open(path).convert("L"))
    return (grey > 0 if grey.max() <= 1 else grey > 127)


def video_of(name: str) -> int:
    m = re.search(r"Video(\d+)", name)
    return int(m.group(1)) if m else -1


def per_image_baseline(mask_dir: Path) -> pd.DataFrame:
    """Metricas por imagen de las dos predicciones triviales."""
    rows = []
    for p in sorted(Path(mask_dir).glob("*.png")):
        gt = read_binary_mask(p)
        fg = float(gt.mean())
        rows.append({
            "video": video_of(p.name), "frame": p.stem, "fraccion_estructura": fg,
            # todo estructura: TP = fg, FP = 1 - fg, FN = 0
            "todo_est_iou": fg,
            "todo_est_dice": 2 * fg / (1 + fg) if fg > 0 else 0.0,
            # todo fondo: TP = 0
            "todo_fondo_iou": 0.0 if fg > 0 else 1.0,
        })
    return pd.DataFrame(rows)


tables = {}
for split in ["training", "validation"]:
    df = per_image_baseline(MMSEG_ROOT / "annotations" / split)
    tables[split] = df
    print(f"--- {split} ({len(df)} frames) --- medias por imagen")
    print(df.groupby("video")[["fraccion_estructura", "todo_est_iou", "todo_est_dice"]]
          .mean().round(4).to_string())
    print(f"GLOBAL  todo estructura: IoU={df.todo_est_iou.mean():.4f}  "
          f"Dice={df.todo_est_dice.mean():.4f}  |  todo fondo: IoU={df.todo_fondo_iou.mean():.4f}\n")

In [ ]:
val = tables["validation"]
base_iou = val.todo_est_iou.mean()
base_dice = val.todo_est_dice.mean()

# Numeros del entrenamiento original. Se leen de Drive si estan, si no se usan los del registro.
summary_path = OUT_DIR / "unet_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print("resumen guardado del baseline:", json.dumps(summary, indent=2), "\n")
unet_iou, unet_dice = 0.7025, 0.7485   # epoca 1, el mejor checkpoint

comparison = pd.DataFrame([
    {"modelo": "U-Net (epoca 1, mejor checkpoint)", "IoU": unet_iou, "Dice": unet_dice},
    {"modelo": "todo estructura (no aprende nada)", "IoU": base_iou, "Dice": base_dice},
    {"modelo": "todo fondo", "IoU": val.todo_fondo_iou.mean(), "Dice": 0.0},
]).set_index("modelo")
comparison.loc["U-Net - todo estructura"] = [unet_iou - base_iou, unet_dice - base_dice]
print(comparison.round(4).to_string())

### 2.1 Resultado de la parte (b)

Ejecutado sobre los datos de Drive el 2026-11-01:

| modelo | IoU | Dice |
|---|---|---|
| U-Net, epoca 1, mejor checkpoint | 0,7025 | 0,7485 |
| todo estructura, no aprende nada | **0,7031** | **0,7490** |
| todo fondo | 0,0000 | 0,0000 |
| **diferencia U-Net - linea base** | **-0,0006** | **-0,0005** |

El U-Net no supera a la linea base trivial: la iguala hasta la cuarta cifra decimal y queda
marginalmente por debajo en las dos metricas.

Esto es mas fuerte que la hipotesis formulada. La hipotesis decia que el modelo *aprende la
constante*; lo medido es que el modelo **es** la constante, sin ventaja residual alguna. Y explica
por que el mejor checkpoint es el de la epoca 1: el modelo colapsa a la clase mayoritaria durante
la primera epoca, y de la 2 a la 40 oscila alrededor de ese punto sin volver a mejorarlo.

Causa de fondo: la fraccion media de estructura en las etiquetas de validacion es **0,703**, y en
10 de los 21 frames supera el 0,93. Las anotaciones de `mmseg_pipes` provienen de
`SAM_Masks_Frames_CLAHE`, cuyas mascaras marcan casi todo el frame. Esto es lo que audita la
task 4.3 (`2_3_sam_mask_audit.ipynb`): en Video6 la estructura real ocupa el 8 % del frame y la
mascara SAM marca el 95 %.

### 2.2 Verificacion con el arnes compartido (task 5.7)

El mismo calculo con el modulo de Anthony, para descartar que el resultado dependa de como se
promedia. Opcional: si la rama no esta disponible, se salta.

In [ ]:
try:
    if not Path("/content/harness/src/evaluation").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b",
                        "feat/labelling-evaluation-tools", REPO_URL, "/content/harness"],
                       check=True)
    sys.path.insert(0, "/content/harness")
    from src.evaluation.segmentation_metrics import evaluate as harness_evaluate

    gts = {p.stem: np.asarray(Image.open(p).convert("L"))
           for p in sorted((MMSEG_ROOT / "annotations" / "validation").glob("*.png"))}
    gts = {k: (v > 0 if v.max() <= 1 else v > 127) for k, v in gts.items()}
    ones = {k: np.ones_like(v) for k, v in gts.items()}
    per_frame, per_video = harness_evaluate(ones, gts, n_boot=500)
    print("linea base 'todo estructura' segun el arnes 5.7:")
    print(per_video[["video", "n_frames", "iou", "dice", "base_iou"]].round(4).to_string(index=False))
    print(f"\nmedia global IoU = {per_frame.iou.mean():.4f}  (seccion 2: {base_iou:.4f})")
except Exception as exc:
    print("verificacion omitida:", type(exc).__name__, exc)

## 3. Auditoria del split

Hallazgo colateral que afecta a la lectura de cualquier curva de validacion. `prepare_mmseg_dataset.py`
y la celda que construye `mmseg_pipes` reparten **frames**, no videos: mezclan todos los pares,
los desordenan y toman el 20 % como validacion. Los frames se extrajeron cada 2 segundos, asi que
dos frames consecutivos del mismo video son casi la misma imagen.

In [ ]:
def frame_index(stem: str) -> int:
    m = re.search(r"frame_(\d+)", stem)
    return int(m.group(1)) if m else -1


split_videos = {s: set(tables[s].video) for s in tables}
for s, vs in split_videos.items():
    print(f"{s:11s} videos={sorted(vs)}  n={len(tables[s])}")
print("videos presentes en AMBOS splits:",
      sorted(split_videos['training'] & split_videos['validation']))

train_keys = {(r.video, frame_index(r.frame)) for r in tables["training"].itertuples()}
neighbours = [(r.video, frame_index(r.frame)) for r in tables["validation"].itertuples()
              if any((r.video, frame_index(r.frame) + d) in train_keys for d in (-1, 1))]
print(f"\nframes de validation con un vecino temporal inmediato en training: "
      f"{len(neighbours)} de {len(tables['validation'])}")
print(" ", neighbours)

### 3.1 Resultado

Los 4 videos con etiquetas (1, 3, 6, 7) aparecen en los dos splits, y **19 de los 21 frames de
validacion tienen un frame inmediatamente anterior o posterior en el conjunto de entrenamiento**.

Consecuencia: el conjunto de validacion no mide generalizacion, mide memorizacion de imagenes casi
identicas. Cualquier curva de validacion de este notebook hay que leerla con eso presente, y la
comparacion entre modelos de la task 4.8 necesita un reparto por video, que es exactamente lo que
define el protocolo *leave-one-video-out* de la task 5.4 de Dayana.

## 4. Parte (a) - curvas de perdida y metricas, train y val

El entrenamiento original solo registraba `train_loss`, `val_miou` y `val_dice`. Faltaban la
perdida de validacion y las metricas sobre entrenamiento, que son las que distinguen sobreajuste
de colapso a la clase mayoritaria. Aqui se reentrena con el bucle instrumentado de
`src/training/loop.py` y el historial se guarda en Drive, para no volver a reentrenar solo para
graficar.

**Requiere GPU.**

In [ ]:
import random

import albumentations as A
import torch
from torch.utils.data import DataLoader

from src.data.pipe_dataset import IGNORE_INDEX, PipeDataset, list_pairs
from src.models.unet import build_unet
from src.training.losses import make_criterion
from src.training.loop import baseline_metrics, evaluate_loader, save_history, train_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| dispositivo", DEVICE)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# Mismas transformaciones que el baseline, para que la comparacion sea valida.
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
])
val_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])

train_pairs = list_pairs(MMSEG_ROOT, "training")
val_pairs = list_pairs(MMSEG_ROOT, "validation")
assert train_pairs and val_pairs, f"no hay pares en {MMSEG_ROOT}"

train_dl = DataLoader(PipeDataset(train_pairs, train_tf), batch_size=BATCH, shuffle=True,
                      num_workers=2, drop_last=False)
val_dl = DataLoader(PipeDataset(val_pairs, val_tf), batch_size=BATCH, shuffle=False,
                    num_workers=2)
print(f"train {len(train_pairs)} | val {len(val_pairs)} | batches {len(train_dl)}/{len(val_dl)}")

print("\nlinea base sobre el loader de validacion (debe coincidir con la seccion 2):")
print(json.dumps(baseline_metrics(val_dl), indent=2))

In [ ]:
set_seed(SEED)
model = build_unet(attention=False, device=DEVICE)
criterion = make_criterion("ce_dice")

result = train_model(model, train_dl, val_dl, criterion, DEVICE,
                     epochs=3 if FAST else EPOCHS, lr=LR, scheduler="cosine",
                     eval_train=True, select_on="dice")

print(f"\nmejor epoca: {result['best_epoch']}  |  mejor val Dice: {result['best_val_dice']:.4f}")
print(f"linea base   : IoU {result['baseline']['todo_estructura_iou']:.4f}  "
      f"Dice {result['baseline']['todo_estructura_dice']:.4f}")
print(f"tiempo: {result['seconds'] / 60:.1f} min")
save_history(result, OUT_DIR / "historial_4_2_base.json")

In [ ]:
def plot_history(result: dict, titulo: str):
    h = pd.DataFrame(result["history"])
    base = result["baseline"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

    axes[0].plot(h.epoch, h.train_loss, label="train", color="#c0392b")
    axes[0].plot(h.epoch, h.val_loss, label="validacion", color="#2471a3")
    axes[0].set_title("perdida (CE + Dice)")
    axes[0].legend()

    axes[1].plot(h.epoch, h.train_iou, label="train", color="#c0392b")
    axes[1].plot(h.epoch, h.val_iou, label="validacion", color="#2471a3")
    axes[1].axhline(base["todo_estructura_iou"], ls="--", color="#555",
                    label=f"todo estructura ({base['todo_estructura_iou']:.3f})")
    axes[1].set_title("IoU de la clase estructura")
    axes[1].legend()

    axes[2].plot(h.epoch, h.train_dice, label="train", color="#c0392b")
    axes[2].plot(h.epoch, h.val_dice, label="validacion", color="#2471a3")
    axes[2].axhline(base["todo_estructura_dice"], ls="--", color="#555",
                    label=f"todo estructura ({base['todo_estructura_dice']:.3f})")
    axes[2].set_title("Dice de la clase estructura")
    axes[2].legend()

    for ax in axes:
        ax.axvline(result["best_epoch"], ls=":", color="#7d3c98", lw=1)
        ax.set_xlabel("epoca")
        ax.grid(alpha=0.25)
    fig.suptitle(f"{titulo} - la linea punteada vertical marca el mejor checkpoint "
                 f"(epoca {result['best_epoch']})", fontsize=11)
    plt.tight_layout()
    plt.show()


plot_history(result, "U-Net, CE + Dice, schedule cosine")

## 5. Parte (c) - variacion de perdida y de schedule

Se cambia **una sola variable** respecto a la configuracion del baseline y se mira si cambia la
epoca del mejor checkpoint. Si el colapso a la constante es la causa, ninguna de estas variaciones
deberia mover el resultado de forma apreciable.

| id | perdida | schedule | que prueba |
|---|---|---|---|
| `control` | CE + Dice | cosine | reproduce el baseline |
| `focal` | Focal (gamma 2, alpha 0,25) | cosine | penalizar los faciles, que aqui son los de la clase mayoritaria |
| `tversky` | Tversky (alpha 0,7, beta 0,3) | cosine | castigar mas los falsos negativos |
| `lr_constante` | CE + Dice | constante | descartar que el cosine sea lo que degrada tras la epoca 1 |

**Requiere GPU.** Son 4 entrenamientos; con 40 epocas y 84 imagenes es del orden de los 4 del
baseline juntos.

In [ ]:
VARIANTES = [
    ("control",      "ce_dice", "cosine"),
    ("focal",        "focal",   "cosine"),
    ("tversky",      "tversky", "cosine"),
    ("lr_constante", "ce_dice", "constant"),
]

resultados = {}
for nombre, perdida, schedule in VARIANTES:
    print(f"\n{'=' * 70}\n{nombre}: perdida={perdida}, schedule={schedule}\n{'=' * 70}")
    set_seed(SEED)
    m = build_unet(attention=False, device=DEVICE)
    r = train_model(m, train_dl, val_dl, make_criterion(perdida), DEVICE,
                    epochs=3 if FAST else EPOCHS, lr=LR, scheduler=schedule,
                    eval_train=True, select_on="dice", log_every=10)
    r["nombre"] = nombre
    r["perdida"] = perdida
    r["schedule"] = schedule
    del r["best_state"]            # no se guardan 4 juegos de pesos
    resultados[nombre] = r
    save_history(r, OUT_DIR / f"historial_4_2_{nombre}.json")

In [ ]:
filas = []
for nombre, r in resultados.items():
    h = pd.DataFrame(r["history"])
    base = r["baseline"]
    filas.append({
        "variante": nombre, "perdida": r["perdida"], "schedule": r["schedule"],
        "mejor_epoca": r["best_epoch"],
        "mejor_val_dice": r["best_val_dice"],
        "val_dice_epoca_1": h.val_dice.iloc[0],
        "val_dice_final": h.val_dice.iloc[-1],
        "base_dice": base["todo_estructura_dice"],
        "mejor_menos_base": r["best_val_dice"] - base["todo_estructura_dice"],
        "mejor_val_iou": h.val_iou.max(),
        "base_iou": base["todo_estructura_iou"],
    })
tabla = pd.DataFrame(filas).set_index("variante")
print(tabla.round(4).to_string())
print("\nSi 'mejor_epoca' sigue siendo 1 o 2 y 'mejor_menos_base' sigue siendo <= 0 en las cuatro,")
print("la causa no esta en la perdida ni en el schedule.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for nombre, r in resultados.items():
    h = pd.DataFrame(r["history"])
    axes[0].plot(h.epoch, h.val_dice, label=nombre)
    axes[1].plot(h.epoch, h.val_iou, label=nombre)
ref = next(iter(resultados.values()))["baseline"]
axes[0].axhline(ref["todo_estructura_dice"], ls="--", color="#555", label="todo estructura")
axes[1].axhline(ref["todo_estructura_iou"], ls="--", color="#555", label="todo estructura")
axes[0].set_title("Dice de validacion por epoca")
axes[1].set_title("IoU de validacion por epoca")
for ax in axes:
    ax.set_xlabel("epoca")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Conclusion

### La hipotesis se acepta, en una version mas fuerte de la enunciada

La hipotesis era: *la etiqueta es casi toda estructura y el modelo aprende la constante.*

Lo medido en la seccion 2, sobre el mismo split del entrenamiento original:

| | IoU | Dice |
|---|---|---|
| U-Net, mejor checkpoint (epoca 1) | 0,7025 | 0,7485 |
| todo estructura | 0,7031 | 0,7490 |
| diferencia | **-0,0006** | **-0,0005** |

El modelo no se aproxima a la constante: **coincide con ella** hasta la cuarta cifra decimal, y
queda marginalmente por debajo. No hay ninguna ganancia residual que atribuir a lo aprendido.

Por que el mejor checkpoint es el de la epoca 1: la unica cosa aprendible con senal suficiente en
estas etiquetas es la clase mayoritaria, y eso se aprende en la primera epoca. De la epoca 2 a la
40 la perdida de entrenamiento sigue bajando (1,12 a 0,81) porque el modelo ajusta la variacion
particular de cada mascara de entrenamiento, pero esa variacion es ruido de etiquetado y no se
traslada a validacion. El Dice de validacion por tanto oscila por debajo del valor de la epoca 1
y no lo supera nunca.

### Evidencia que lo respalda

1. **La fraccion de estructura de las etiquetas es 0,703 en validacion y 0,684 en entrenamiento.**
   En 10 de los 21 frames de validacion supera el 0,93. Son etiquetas en las que la constante es
   casi optima por construccion.
2. **Las etiquetas no son fiables.** `mmseg_pipes` se construye desde `SAM_Masks_Frames_CLAHE`. La
   task 4.3 (`2_3_sam_mask_audit.ipynb`) mide esas mascaras contra el ground truth curado: en
   Video1 dan IoU 0,664 frente a una base de 0,718, y en Video6 dan IoU 0,083 marcando el 95 % del
   frame cuando la estructura real ocupa el 8 %. El modelo esta ajustando ruido.
3. **El split no mide generalizacion** (seccion 3): los 4 videos estan en los dos lados y 19 de 21
   frames de validacion tienen un vecino temporal inmediato en entrenamiento.
4. **La causa no esta en la perdida ni en el schedule** (seccion 5). Se confirma al ejecutar las
   cuatro variantes: si `mejor_epoca` se mantiene en 1 o 2 y ninguna variante supera la linea
   base, la eleccion de perdida es irrelevante mientras la etiqueta sea la que es.

### Lo que no concluye este notebook

Nada sobre la calidad del U-Net como arquitectura. Con estas etiquetas el experimento no puede
distinguir un modelo bueno de uno malo, porque el techo medible es la constante. La pregunta
"sirve el U-Net para esto" solo se puede responder cuando la task 5.1 complete las mascaras
curadas de los 7 videos y `mmseg_pipes` se regenere desde `Test_Set_Curado`.

### Acciones que se derivan

1. **Regenerar `mmseg_pipes` desde `Test_Set_Curado`** en vez de desde las mascaras SAM. Hoy hay
   28 mascaras curadas (Video1 con 20 y Video6 con 8) de las 68 previstas en la task 5.1.
2. **Repartir por video, no por frame**, al construir el split. Coordinar con el protocolo
   *leave-one-video-out* de la task 5.4 para que la task 4.8 sea comparable.
3. **Reportar siempre la linea base al lado del modelo.** Sin ella, un Dice de 0,7485 parece un
   resultado y es un empate con no hacer nada. El arnes de la task 5.7 ya lo hace por defecto.
4. La ablacion de la task 5.11 (`3_3_unet_ablation.ipynb`) no debe interpretarse hasta que (1) y
   (2) esten hechos: sobre estas etiquetas mediria la misma constante cuatro veces.